# Predators and prey

summer4 is a library for any system in which things move between compartments, not only for
epidemics. Ecology, pharmacokinetics (a drug moving between blood and tissue) and chemistry
(molecules turning into other molecules) are all the same kind of model. This page builds a
predator-prey model, the Lotka-Volterra model, to show that the tools from the previous pages
work unchanged when the compartments are animals instead of patients.

There are two compartments: `prey` (say, rabbits) and `predator` (say, foxes). Textbooks
usually write the model as one gain and one loss per species. Here we **unfold** it, with a
separately named flow for each thing that actually happens:

- **prey birth**: new prey arrive, in proportion to how many prey there are;
- **prey natural death**: each prey animal dies at a fixed rate;
- **predation**: prey are eaten, at a rate that grows with the number of predators;
- **predator birth**: new predators appear, in proportion to the amount of prey eaten;
- **predator natural death**: each predator dies at a fixed rate.

By the end you will have plotted the two populations over time, plotted them against each
other, plotted every flow on its own, and checked that the unfolded model gives the same
answer as the classic two-term version.

## Imports

The next cell loads the libraries, as on the earlier pages. Nothing is drawn yet.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    EntryFlow,
    ExitFlow,
    FlowMass,
    FlowModel,
    FlowRef,
    OutputSet,
    Param,
    Property,
    PropertyMap,
    Reduce,
    SavePlan,
    SaveRequest,
)

## Compartments

The property is `species` and its values are `prey` and `predator`. The cell prints the two
compartment labels.

In [ ]:
species = Property("species", ("prey", "predator"))
pmap = PropertyMap.from_property(species)
pmap.labels()

## Flows that need the number of animals

Two of the rates depend on how many animals there are right now: prey birth depends on the
number of prey, and predation depends on the number of predators. summer4 writes "the number of
animals in some compartments" as `Reduce(where=...)`. In plain words, it adds up all the
animals in the compartments that match the selector and gives one number. So
`Reduce(where=species["predator"])` is the number of predators, and `Param("a") * Reduce(where=species["predator"])`
is "a times the number of predators". `Reduce()` with no selector is the whole population.

## The prey flows

`EntryFlow(name, destination, rate)` adds animals from outside the model, at a rate in animals
per day. Prey birth is `alpha * prey`, so the rate is `alpha` times the number of prey.

`ExitFlow(name, source, rate)` removes animals at a rate **per animal**. Natural death takes a
fixed fraction `mu` of the prey each day. Predation takes `a * predators` of each prey animal
per day, so a bigger predator population eats a bigger share of the prey.

The cell builds the model and adds these three flows.

In [ ]:
model = FlowModel(pmap)
model.add_flow(EntryFlow("prey_birth", species["prey"], Param("alpha") * Reduce(where=species["prey"])))
model.add_flow(ExitFlow("prey_death", species["prey"], Param("mu")))
predation = model.add_flow(
    ExitFlow("predation", species["prey"], Param("a") * Reduce(where=species["predator"]))
)

## The predator flows, tied to predation

Predators are born because they eat. A fixed fraction `e` (the efficiency) of the prey eaten
turns into new predators. Rather than write the predation rate out a second time, the predator
birth flow **refers to the predation flow itself**: `FlowRef("predation")` stands for the
number of prey eaten per day, and `.sum_by()` adds it up over the whole model to a single
number. So `e * FlowRef("predation").sum_by()` reads as "efficiency times prey eaten". If you
later change how predation works, predator births follow automatically.

Predator death is an ordinary exit flow at rate `d`. The cell adds both flows, and then sets
the starting populations: 40 prey and 15 predators.

In [ ]:
predator_birth = model.add_flow(
    EntryFlow("predator_birth", species["predator"], Param("e") * FlowRef("predation").sum_by())
)
model.add_flow(ExitFlow("predator_death", species["predator"], Param("d")))
start = model.set_initial_population({species["prey"]: 40.0, species["predator"]: 15.0})

## Compile and run

The parameters are all per day. With these values the populations balance at 25 prey and
25 predators, and starting away from that point makes them cycle around it.

The default solver takes fixed steps and slowly spirals outwards on this model, because each
step slightly overshoots. For a clean closed cycle we ask for the adaptive solver `"dopri5"`
and a tight tolerance (`rtol` and `atol` of `1e-8`). If your own cycles drift, tighten these
first.

We save the number of animals in each compartment, four times a day, for 100 days.

In [ ]:
compiled = model.compile()
params = {"alpha": 0.6, "mu": 0.1, "a": 0.02, "e": 0.5, "d": 0.25}
plan = SavePlan(
    requests={"compartments": SaveRequest(Compartments())},
    ts=np.linspace(0.0, 100.0, 401),
)
result = compiled.run(
    params, t0=0.0, t1=100.0, dt=0.1, save=plan, solver="dopri5", rtol=1e-8, atol=1e-8
)
frame = result["compartments"].to_pandas()
frame.columns = ["prey", "predator"]

## Populations over time

The plot should show two repeating waves, about five cycles in 100 days. The predator wave
trails the prey wave: predators peak after the prey, because they need prey to eat before they
can breed, and the prey recover once the predators have thinned out. Each cycle should reach
the same height as the one before.

In [ ]:
frame.plot(
    title="Prey and predators over time",
    labels={"index": "time (days)", "value": "animals", "variable": ""},
)

## The phase plane

A phase-plane plot draws the number of predators against the number of prey, with time left
implicit. A model that repeats the same cycle forever traces a closed loop. The plot should
show a single closed loop around the balance point (25 prey, 25 predators), travelled
anticlockwise: the prey rise, then the predators rise, then the prey fall, then the
predators fall. If the solver drifts, the loop becomes a spiral instead.

In [ ]:
frame.plot(
    x="prey",
    y="predator",
    title="Phase plane: predators against prey",
    labels={"prey": "prey (animals)", "predator": "predators (animals)"},
)

## Look at each flow separately

This is what unfolding buys. Each flow has a name, so we can ask for it. `FlowMass(name)` is
the number of animals moving along a flow each day, and `.sum_by()` adds it up to one number
per day. An `OutputSet` collects the outputs we want, as on the page on derived outputs.

The first plot shows the three flows that touch the prey: births, natural deaths and
predation. Births and natural deaths should rise and fall with the prey population (births are six
times deaths, since `alpha` is six times `mu`). Predation should rise and fall too, but it
peaks later than prey births, because it depends on the predators as well. The second plot
shows the predator flows. Predator births should look exactly like predation scaled by `e`
(0.5), because that is how we defined them.

In [ ]:
outputs = OutputSet()
for flow in ("prey_birth", "prey_death", "predation", "predator_birth", "predator_death"):
    outputs[flow] = FlowMass(flow).sum_by()

flow_result = compiled.run(
    params,
    t0=0.0,
    t1=100.0,
    dt=0.1,
    save=outputs.plan(SavePlan(ts=np.linspace(0.0, 100.0, 401))),
    solver="dopri5",
    rtol=1e-8,
    atol=1e-8,
)
flows = outputs.evaluate(flow_result, params)

flows.to_frame(
    names=["prey_birth", "prey_death", "predation"], backend="pandas"
).plot(
    title="Flows into and out of the prey",
    labels={"time": "time (days)", "value": "animals per day", "variable": ""},
)

In [ ]:
flows.to_frame(names=["predator_birth", "predator_death"], backend="pandas").plot(
    title="Flows into and out of the predators",
    labels={"time": "time (days)", "value": "animals per day", "variable": ""},
)

## Compare with the classic two-term model

The usual textbook version has one gain and one loss for each species. We build it as well, so
that you can compare:

- prey gain `alpha * prey`, prey loss `mu + a * predators` per animal;
- predator gain `e * a * predators * prey`, predator loss `d` per animal.

It is the same mathematics with the flows merged, so the trajectories should coincide. The
plot overlays the prey numbers from the two versions (the lines lie on top of each other), and
the cell below it prints the largest difference between the two models in either species.

In [ ]:
lumped = FlowModel(pmap)
lumped.add_flow(EntryFlow("prey_gain", species["prey"], Param("alpha") * Reduce(where=species["prey"])))
lumped.add_flow(
    ExitFlow(
        "prey_loss",
        species["prey"],
        Param("mu") + Param("a") * Reduce(where=species["predator"]),
    )
)
lumped.add_flow(
    EntryFlow(
        "predator_gain",
        species["predator"],
        Param("e")
        * Param("a")
        * Reduce(where=species["predator"])
        * Reduce(where=species["prey"]),
    )
)
lumped.add_flow(ExitFlow("predator_loss", species["predator"], Param("d")))
lumped_start = lumped.set_initial_population({species["prey"]: 40.0, species["predator"]: 15.0})

lumped_result = lumped.compile().run(
    params, t0=0.0, t1=100.0, dt=0.1, save=plan, solver="dopri5", rtol=1e-8, atol=1e-8
)
lumped_frame = lumped_result["compartments"].to_pandas()
lumped_frame.columns = ["prey", "predator"]

pd.DataFrame({"unfolded": frame["prey"], "lumped": lumped_frame["prey"]}).plot(
    title="Prey: unfolded model and classic two-term model",
    labels={"index": "time (days)", "value": "prey (animals)", "variable": ""},
)

In [ ]:
float((frame - lumped_frame).abs().to_numpy().max())

The difference is at rounding level, effectively zero. The classic version gives the same
populations, but it cannot tell you how many animals were eaten each day, because predation
has been merged into the loss term. In the unfolded model that number is just
`FlowMass("predation")`.

## Things to try

- Change `e` to 0.25 and rerun. The balance point for prey is `d / (e * a)`, so it moves.
  Which cells change, and how does the phase-plane loop move?
- Start at the balance point exactly (`prey` 25, `predator` 25). What do the populations do?
- Add an `ExitFlow("harvest", species["prey"], Param("h"))` for hunting, add `"h": 0.1` to
  `params`, and watch the flow plot. Which flows change, and which stay the same?
- Rerun the first run with the default solver (leave out `solver`, `rtol` and `atol`) and
  `dt=0.01`. Does the phase-plane loop close?
- Add a prey crowding loss, `ExitFlow("crowding", species["prey"], Param("c") * Reduce(where=species["prey"]))`,
  with `"c": 0.002` in `params`. What happens to the cycles? Do they still repeat?